In [ ]:
## Nikolay Vorontsov, 23.11.2024, Prompting LLMs with validation set, ask to find hallucinations and label them.
## Model used: gemini-1.5-flash
## required files: llabel_validation_set_with_llm_for_baseline.env
##                 mushroom.en-val.v2.unlabeled.jsonl

## UPDATED: CHANGED TO VALIDATE TEST SET
## required files:

In [1]:
#INSTALL DEPENDENCIES

!pip install openai

In [2]:
# IMPORT LIBRARIES

import configparser
import json
from google.colab import userdata

from google.colab import drive
drive.mount('/content/drive')

import time
from openai import OpenAI


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# DEFINE VARIABLES
client = OpenAI(api_key = userdata.get('GPT_MUSHROOM'))
model_used = "gpt-4o-mini" #"gpt-4o-2024-08-06"

prompts = configparser.ConfigParser()

lang = ["ar", "ca", "cs", "de", "en", "es", "eu", "fa", "fi", "fr", "hi", "it", "zh"]
# /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/v1/mushroom.{lang[5]}-tst.v1.jsonl
prompts.read('/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_GPT_v2.env')

set_to_label =  f'/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/v1/mushroom.{lang[11]}-tst.v1.jsonl'
output_file=f"/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/final_submission_LLM/mushroom.{lang[11]}-tst.v1_{model_used}_v2.jsonl"


In [4]:
lang[11]

'it'

In [5]:
# "a" creates the file if it doesn't exist
with open(output_file, "a") as file:
    pass  # Do nothing, just ensure the file exists

print(f"{output_file} is created or already exists.")

/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/final_submission_LLM/mushroom.it-tst.v1_gpt-4o-mini_v2.jsonl is created or already exists.


In [6]:
# Function to load the .jsonl file and parse its contents
def load_jsonl(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        # Read each line and parse as a JSON object
        return [json.loads(line) for line in file]

# Load the data
data = load_jsonl(set_to_label)


In [18]:
#FUNCTIONS

In [7]:
# COMPILE PROMPTS
def define_prompt(datapoint):

  #samples can be also imported from a jsonl file.
  Sample1 = prompts.get('SAMPLES', 'Sample1')

  prompt1 = (
      f"{prompts.get('PROMPTS', 'p0')}"
      f"{datapoint}"
      f"{prompts.get('PROMPTS', 'p1')}"
      f"{prompts.get('PROMPTS', 'p2')}"
      f"{prompts.get('PROMPTS', 'p3')}"
      f"{prompts.get('PROMPTS', 'p4')}"
      f"{prompts.get('PROMPTS', 'p5')}"
      )

  return prompt1

In [8]:
define_prompt(data[1])

'This json line contains output from an llm, question-answer pair in "model_input", "model_output_text".{\'id\': \'tst-it-2\', \'lang\': \'IT\', \'model_input\': \'Quali colorazioni varia il plasmodio del Fuligo septica?\', \'model_output_text\': \'Il plasmodio del Fuligo septica varia in colore da giallo pallido a giallo-verdastro, con una tonalità verdognola-giallastra.\', \'model_id\': \'sapienzanlp/modello-italia-9b\', \'model_output_tokens\': [\'Il\', \'▁pla\', \'smo\', \'dio\', \'▁del\', \'▁Fu\', \'li\', \'go\', \'▁sep\', \'tica\', \'▁varia\', \'▁in\', \'▁colore\', \'▁da\', \'▁giallo\', \'▁palli\', \'do\', \'▁a\', \'▁giallo\', \'-\', \'ver\', \'da\', \'stro\', \',\', \'▁con\', \'▁una\', \'▁tonalità\', \'▁ver\', \'do\', \'gnola\', \'-\', \'gia\', \'lla\', \'stra\', \'.\', \'</s>\'], \'model_output_logits\': [179.0, 262.0, 190.0, 188.0, 292.0, 238.0, 185.0, 144.0, 171.0, 143.0, 204.0, 290.0, 177.0, 352.0, 148.0, 198.0, 209.0, 286.0, 143.0, 155.0, 108.5, 227.0, 175.0, 255.0, 203.0, 

In [9]:
def process_data(datapointX):
    selected_prompt = define_prompt(datapointX)
    completion = client.chat.completions.create(
        model=f"{model_used}",
        messages=[{"role": "user", "content": selected_prompt}]
        )
    hallucinated_words = [list_element.strip("- ") for list_element in completion.choices[0].message.content.split("\n") if list_element] ## this ensure list_element is not empty
    return hallucinated_words

In [12]:
## TEST process_data(datapoint)
process_data(data[50])

['cappuccio rosso']

In [13]:
def find_spans(datapointX, hallucinated_words):

  spans = []
  model_output_text = datapointX["model_output_text"]

  for word in hallucinated_words:
      # Initialize the starting index for each word search
      start_index = 0
      while True:
          start_index = model_output_text.find(word, start_index)
          if start_index == -1:
              break
          end_index = start_index + len(word)
          spans.append([start_index, end_index])
          # Move the starting index past the current word to avoid overlapping results
          start_index = end_index

  return spans


In [14]:
## Assessing the output file

def last_line_number(file_path):
  # Read the last line of the file
  last_line = None
  with open(file_path, "r") as file:
      for line in file:
          last_line = line.strip()  # Store the current line

  # Parse the JSON object from the last line
  if last_line:
      last_data = json.loads(last_line)
      #print("Last JSON object:", last_data)
      return last_data["number"]
  else:
      print("The file is empty")
      return None


In [15]:
##TESTING
last_line_number(output_file)

114

In [16]:
for key, value in data[0].items():
  print(f'"{key}":') #type(value))

"id":
"lang":
"model_input":
"model_output_text":
"model_id":
"model_output_tokens":
"model_output_logits":


In [17]:
def label_and_save_data(data):
    processed_count = 0  # Counter for newly processed entries

    for number, datapoint in enumerate(data, start=1):
        # Get the last processed ID from the output file
        last_processed_number = last_line_number(output_file) or 0
        #print("Last processed ID:", last_processed_number)

        # Skip already processed entries
        if number <= last_processed_number:
            continue

        # Define prompt and process data
        prompt = define_prompt(datapoint)
        hallucinated_words = process_data(datapoint)
        hard_labels = find_spans(datapoint, hallucinated_words)

        # Save the datapoint to the JSONL file
        with open(output_file, "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "number": number,
                "id": datapoint["id"],
                "lang": datapoint["lang"],
                "model_input": datapoint["model_input"],
                "model_output_text": datapoint["model_output_text"],
                "model_id": datapoint["model_id"],
                "hallucinated_words": hallucinated_words,
                "soft_labels": [],
                "hard_labels": hard_labels,
                "model_output_logits": datapoint["model_output_logits"],
                "model_output_tokens": datapoint["model_output_tokens"],
            }

            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")
        print(datapoint["model_output_text"])
        print(hallucinated_words)
        print(hard_labels)
        print("-------------------------------")

        # Increment the processed count
        processed_count += 1

        # Stop processing after 5 new entries, wait for 10 seconds
        #if processed_count >= 5:
        #    print(f"Processed {processed_count} entries. Waiting for 10 sec. at ID {number}.")
        #    time.sleep(10)

        #    processed_count = 0


In [18]:
label_and_save_data(data)

La Supercoppa UEFA 2011 fu disputata tra il Real Madrid, vincitore della UEFA Champions League 2010-2011, e il Chelsea, vincitore della UEFA Europa League 2010-2011.
['La', 'Champions', 'League', '2010-2011', 'Europa', 'League', '2010-2011']
[[0, 2], [78, 87], [88, 94], [148, 154], [95, 104], [155, 164], [141, 147], [88, 94], [148, 154], [95, 104], [155, 164]]
-------------------------------
Moritz Volz ha fatto 249 presenze in campionato con l'Arsenal Football Club.
['Moritz Volz', '249', "l'Arsenal", 'Football', 'Club']
[[0, 11], [21, 24], [52, 61], [62, 70], [71, 75]]
-------------------------------
La collina Aukštojas si trova nel comune di Jelgava, in Lettonia.
['La collina Aukštojas si trova nel comune di Jelgava, in Lettonia.', 'Aukštojas', 'Jelgava']
[[0, 65], [11, 20], [44, 51]]
-------------------------------
Il generale tedesco Adolf Joseph Ferdinand Galland morì il 5 febbraio 1942, durante una missione per distruggere una base navale britannica a Scapa Flow, Isole Orcadi, 

In [19]:
output_file

'/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/final_submission_LLM/mushroom.it-tst.v1_gpt-4o-mini_v2.jsonl'

In [20]:
## Resave a copy with no extra keys.

with open(output_file, "r", encoding='utf-8') as jsonl_file:
    lines = jsonl_file.readlines()

    for line in lines:

        # Remove '_unlabeled' from the 'id' field
        data_to_resave = json.loads(line)

        data_to_resave['id'] = data_to_resave['id'].replace('_unlabeled', '')
        print(data_to_resave['id'])

        print(data_to_resave["hard_labels"])

        soft_labels = [{'start': label[0], 'prob': float(1), 'end': label[1]} for label in data_to_resave['hard_labels'] if label]
        print(soft_labels)

        # Save the datapoint to the JSONL file
        with open(f"{output_file}_no_extra_keys_soft_labels_prob1.jsonl", "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "id":data_to_resave["id"],
                "lang":data_to_resave["lang"],
                "model_input":data_to_resave["model_input"],
                "model_output_text":data_to_resave["model_output_text"],
                "model_id":data_to_resave["model_id"],
                "soft_labels":soft_labels, #instead of data_to_resave["soft_labels"], that is to output an empty list.
                "hard_labels":data_to_resave["hard_labels"],
                "model_output_logits":data_to_resave["model_output_logits"],
                "model_output_tokens":data_to_resave["model_output_tokens"],
            }
            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")



tst-it-1
[[0, 2], [47, 60], [61, 70], [71, 76], [77, 79], [79, 81]]
[{'start': 0, 'prob': 1.0, 'end': 2}, {'start': 47, 'prob': 1.0, 'end': 60}, {'start': 61, 'prob': 1.0, 'end': 70}, {'start': 71, 'prob': 1.0, 'end': 76}, {'start': 77, 'prob': 1.0, 'end': 79}, {'start': 79, 'prob': 1.0, 'end': 81}]
tst-it-2
[[68, 84], [94, 102], [103, 124]]
[{'start': 68, 'prob': 1.0, 'end': 84}, {'start': 94, 'prob': 1.0, 'end': 102}, {'start': 103, 'prob': 1.0, 'end': 124}]
tst-it-3
[[0, 2], [7, 10], [10, 13], [18, 19], [19, 24], [26, 28], [28, 33], [25, 26], [41, 42], [51, 53], [69, 70]]
[{'start': 0, 'prob': 1.0, 'end': 2}, {'start': 7, 'prob': 1.0, 'end': 10}, {'start': 10, 'prob': 1.0, 'end': 13}, {'start': 18, 'prob': 1.0, 'end': 19}, {'start': 19, 'prob': 1.0, 'end': 24}, {'start': 26, 'prob': 1.0, 'end': 28}, {'start': 28, 'prob': 1.0, 'end': 33}, {'start': 25, 'prob': 1.0, 'end': 26}, {'start': 41, 'prob': 1.0, 'end': 42}, {'start': 51, 'prob': 1.0, 'end': 53}, {'start': 69, 'prob': 1.0, 'en